## Data utilities
this file loads all datasets and contains helper functions for cleaning, validation, standardising columns and creating image grids

the cifar-10 cells run anywhere, they download the data.
the flickr8k, tweet sentiment and samsum cells need local copies of those datasets, each section says which files

In [ ]:
# data and i/o
import os
import re
import sys
import json
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np

# dataset loading & handling
from datasets import load_dataset, Image as HFImage
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from pandas.api.types import is_integer_dtype

# plotting
import matplotlib.pyplot as plt

# image handling
from PIL import Image as PILImage
from random import Random

# transformers and torch, predict_sentiment_labels needs torch.no_grad and torch.argmax
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score
from torch.nn.functional import softmax

### where files are written

the cifar-10 loader writes 50,000 pngs, they go to `KD_DATA_DIR` (default `~/.cache/kd-data`) instead of the folder the notebook runs in

In [ ]:
# downloads and generated images land outside the repository, set KD_DATA_DIR to put them somewhere else
DATA_DIR = Path(os.environ.get('KD_DATA_DIR', Path.home() / '.cache' / 'kd-data'))
print(f"data directory: {DATA_DIR}")

### Helper functions
- `preprocess text` (sentiment analysis)
- `map_labels`
- `train_val_test_split`
- `save_df (I/O)`
- `load_df (I/O)`
- `show_basic_stats`
- `plot class counts`
- `show_image_grid`
- `standardise_columns`
- `load_sentiment_data`
- `load_model_and_tokenizer`
- `predict_sentiment_labels`
- `evaluate`

text cleaning for the sentiment data

In [ ]:
# strip urls and punctuation from sentiment text before tokenizing
def preprocess_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = text.lower()
    return text

csv input and output

In [ ]:
def save_df(df: pd.DataFrame, path: str | Path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

def load_df(path: str | Path) -> pd.DataFrame:
    return pd.read_csv(path)

stratified train, validation and test split, and label mapping

In [ ]:
def train_val_test_split(df: pd.DataFrame,
                         label_col: str,
                         test: float = 0.2,
                         val: float = 0.1,
                         seed: int = 42):
    train_df, temp_df = train_test_split(
        df, test_size=test+val, stratify=df[label_col], random_state=seed)
    val_size = val / (test + val)
    val_df, test_df = train_test_split(
        temp_df, test_size=val_size, stratify=temp_df[label_col], random_state=seed)
    return train_df, val_df, test_df

# map raw label values to class ids, values missing from mapping become nan
def map_labels(df: pd.DataFrame, col: str, mapping: dict) -> pd.DataFrame:
    df[col] = df[col].map(mapping)
    return df

quick looks at a dataframe

In [ ]:
def show_basic_stats(df: pd.DataFrame):
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("First 3 rows:\n", df.head(3))
    for col in df.select_dtypes(include='object').columns:
        print(f"Unique values in '{col}':", df[col].nunique())

def plot_class_counts(df: pd.DataFrame, label_col: str):
    df[label_col].value_counts().sort_index().plot(kind="bar")
    plt.title("Class distribution")
    plt.xlabel(label_col)
    plt.ylabel("Count")
    plt.show()

def show_image_grid(df: pd.DataFrame, path_col: str, n: int = 6, seed: int = 0):
    # a seeded generator, so a re-run shows the same images
    paths = Random(seed).sample(df[path_col].tolist(), n)
    fig, axes = plt.subplots(1, n, figsize=(n*2, 2))
    for ax, p in zip(axes, paths):
        ax.imshow(PILImage.open(p))
        ax.axis("off")
    plt.show()

column renaming

In [ ]:
def standardise_columns(df: pd.DataFrame, rename_dict: dict) -> pd.DataFrame:
    return df.rename(columns=rename_dict)

sentiment csv and a hugging face classifier

In [ ]:
def load_sentiment_data(csv_path: str, text_col: str = "text", label_col: str = "label"):
    df = pd.read_csv(csv_path)
    return df[text_col].tolist(), df[label_col].tolist()


def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()
    return model, tokenizer

predicted class ids for a list of texts

In [ ]:
def predict_sentiment_labels(model, tokenizer, texts):
    predictions = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = softmax(logits, dim=1)
            pred = torch.argmax(probs).item()
            predictions.append(pred)
    return predictions

accuracy and macro f1, written to a json file

In [ ]:
# macro f1 weighs every class equally, the binary default would reject the three class sentiment labels
def evaluate(model_name, y_true, y_pred, output_json, average="macro"):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average=average)

    results = {
        "model": model_name,
        "task": "sentiment",
        "metrics": {
            "accuracy": round(acc, 4),
            f"f1_{average}": round(f1, 4)
        },
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    # a bare file name has no folder part, and makedirs fails on an empty path
    os.makedirs(os.path.dirname(output_json) or ".", exist_ok=True)
    with open(output_json, "w") as f:
        json.dump(results, f, indent=2)

    print("Evaluation complete")
    print(json.dumps(results, indent=2))
    return results

## load all datasets into separate dataframes

cifar-10 from hugging face, each image saved as a png and listed with its label

In [ ]:
def load_cifar10_df(root_dir: str | Path = DATA_DIR / "cifar10_images",
                    split: str = "train",
                    save_png: bool = True) -> pd.DataFrame:
    """
     returns a dataframe with columns:  image_path | label
    """
    cifar = load_dataset("cifar10", split=split)        
    cifar = cifar.cast_column("img", HFImage(decode=True))  

    paths, labels = [], []
    out_dir = Path(root_dir) / split
    out_dir.mkdir(parents=True, exist_ok=True)

    for i, row in tqdm(enumerate(cifar), total=len(cifar), desc="Saving PNGs"):
        # images decoded from the hugging face cache have an empty filename, so the path is always
        # the png this function writes, with save_png off it only reuses the files an earlier call wrote
        file_path = out_dir / f"{i:05d}.png"
        if save_png:
            row["img"].save(file_path)
        paths.append(file_path.as_posix())
        labels.append(row["label"])

    return pd.DataFrame({"image_path": paths, "label": labels})

humaneval, the openai code generation benchmark, as a dataframe

In [ ]:
def load_humaneval_df() -> pd.DataFrame:
    """
    returns a dataframe with columns:  task_id | prompt | canonical_solution | test
    """
    he = load_dataset("openai_humaneval", split="test")
    return he.to_pandas()[["task_id", "prompt", "canonical_solution", "test"]]

## exploration of each dataset loaded:

### cifar-10

downloads cifar-10 from hugging face on the first run, no local data needed

In [ ]:
cifar_df = load_cifar10_df(split="train")   
print("Loaded CIFAR-10 ->", len(cifar_df), "rows")

check that the image files exist and the labels are the integers 0 to 9

In [ ]:
# sanity checks on the saved images and labels
first_path = cifar_df.loc[0, "image_path"]
assert os.path.exists(first_path), f"File missing: {first_path}"
assert "label" in cifar_df.columns
assert is_integer_dtype(cifar_df["label"]), "Labels are not integer dtype"
assert cifar_df["label"].between(0, 9).all(), "Labels outside 0-9"
print(" first image exists at", first_path)
print("dataFrame is valid, labels are integers 0-9")

size, columns and class balance, then a few random images

In [ ]:
# basic stats and class counts
show_basic_stats(cifar_df)

In [ ]:
plot_class_counts(cifar_df, "label")

In [ ]:
show_image_grid(cifar_df, "image_path", n=6)

### flickr8k captions

needs a local copy of flickr8k: `datasets/captioning/captioning_text/Flickr8k.token.txt` and the images in `datasets/captioning/captioning_dataset/Flicker8k_Dataset/`, relative to the folder the notebook runs in.
these cells were not run for the saved version of this notebook because that data is not available here, so they have no outputs

In [ ]:
# flickr8k token file: one caption per line, "<image>#<n>\t<caption>", five captions per image
def load_captions(filepath):
    captions_dict = {}
    with open(filepath, 'r') as f:
        for line in f:
            img_id, caption = line.strip().split('\t')
            img_id = img_id.split('#')[0]
            if img_id not in captions_dict:
                captions_dict[img_id] = []
            captions_dict[img_id].append(caption)
    return captions_dict

load the captions, then show the first image with its five captions

In [ ]:
captions_path = 'datasets/captioning/captioning_text/Flickr8k.token.txt'
captions = load_captions(captions_path)
print(f"Loaded {len(captions)} images with captions")

In [ ]:
sample_id = list(captions.keys())[0]
img_path = os.path.join('datasets/captioning/captioning_dataset/Flicker8k_Dataset', sample_id)
img = PILImage.open(img_path)
plt.imshow(img)
plt.axis('off')
plt.title('\n'.join(captions[sample_id]))
plt.show()

### tweet sentiment

needs the test split of the kaggle tweet sentiment dataset (`sentiment-analysis-dataset`, version 9) at the machine specific path in the next cell, change it to wherever `test.csv` lives.
not run for the saved version of this notebook, so these cells have no outputs

In [ ]:
# local copy of the kaggle tweet sentiment test set, the path is machine specific
sentiment_df = pd.read_csv("e:/ML/datasets/sentiment-analysis-dataset/versions/9/test.csv", delimiter=",", encoding="latin1")

In [ ]:
# sentiment analysis dataset exploration
sentiment_df.info()
print(sentiment_df.shape)

map the sentiment names to class ids

In [ ]:
# three class sentiment ids
SENTIMENT_MAP = {"negative": 0, "neutral": 1, "positive": 2}
# rows missing a text or a sentiment leave both lists together, so the i-th text and the i-th label
# still describe the same tweet
labelled = sentiment_df.dropna(subset=["text", "sentiment"])
TEXTS = labelled["text"].tolist()
LABELS = [SENTIMENT_MAP[label.lower()] for label in labelled["sentiment"]]

assert len(TEXTS) == len(LABELS)
print(f"{len(TEXTS)} labelled tweets, {len(sentiment_df) - len(labelled)} empty rows dropped")
print("Mapped labels:", LABELS[:5])

### samsum dialogue summarization

needs `samsum-test.csv` from the kaggle samsum dataset under `datasets/samsum-dataset-text-summarization/versions/1/`, relative to the folder the notebook runs in.
not run for the saved version of this notebook, so these cells have no outputs

In [ ]:
# local copy of the samsum test set, the path is relative to the folder the notebook runs in
textSummarization_df = pd.read_csv(r"datasets\samsum-dataset-text-summarization\versions\1\samsum-test.csv", delimiter=",", encoding="latin1")

In [ ]:
# text summarization dataset exploration
# only the last expression of a cell is displayed, so the shape and the columns are printed
textSummarization_df.info()
print(textSummarization_df.shape)
print(list(textSummarization_df.columns))